In [27]:
import os
from dotenv import load_dotenv
from pprint import pprint

import pandas as pd

import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings

import google.generativeai as genai

from IPython.display import Markdown

In [28]:


load_dotenv()

api_key = os.getenv('GEMENI_API_KEY')
#print(api_key)
genai.configure(api_key=api_key)




In [29]:
for a in genai.list_models():
    if 'embedContent' in a.supported_generation_methods:
        print(a.name)

models/embedding-001
models/text-embedding-004
models/gemini-embedding-exp-03-07
models/gemini-embedding-exp
models/gemini-embedding-001


In [30]:
import json

with open(r'C:\Users\liali\YoloWdTagger\wdv3-timm\tag_desctiptions\eye_tags.json') as f:
    data= json.load(f)

print(data)

[{'instruction': 'ringed eyes', 'input': '', 'output': 'Eyes in which the pupil and iris are composed of one or several rings or circles, usually concentric to the pupil. Eyes composed of many of these rings are frequently used to give a crazed appearance.'}, {'instruction': 'aqua eyes', 'input': '', 'output': 'A character with blue-green colored eyes. Due to the subjective nature of color judgment and other factors (such as the lighting on the characters) there is an overlap between this tag, the blue eyes, and the green eyes tags.'}, {'instruction': 'black eyes', 'input': '', 'output': 'A character with black colored eyes. See bruised eye for eyes that are bruised, also known as a black eye. Also see solid circle pupils for eyes with irises that are completely flat black to the point of being indistinct from the pupil.'}, {'instruction': 'blue eyes', 'input': '', 'output': 'A character with blue colored eyes.Due to the subjective nature of color judgment and other factors (such as th

In [31]:


documents = []

for item in data:
    entry = ""
    if item['instruction'] != '':
        entry += f"Instruction : {item['instruction']}\n"

    if item['input'] != '':
        entry += f"Input : {item['input']}\n"

    if item['output'] != '':
        entry += f"Output : {item['output']}"

    documents.append(entry)

len(documents)



12

In [32]:
from PyPDF2 import PdfReader

def extract_text_from_pdf(file_path):
    pdf_reader = PdfReader(file_path)
    num_pages = len(pdf_reader.pages)

    page_offset = 0
    text = ""

    for page in range(page_offset, num_pages):
        text += pdf_reader.pages[page].extract_text()

    return text


text = extract_text_from_pdf(r'C:\Users\liali\YoloWdTagger\wdv3-timm\pdf_files\Tag Group_Eyes Tags Wiki _ Danbooru.pdf')
print(text)

Danbooru
My Account Posts Comments Notes Artists Tags Pools WikiForum
More »Search wiki pagesSearch New Changes Help |Posts (0) History Edit
Recent Changes ( all)
cheval grand (summer calm
navy drop) (umamusume)
saltire
spoon bending
til arrior
minoyama
rondoline e. effenberg (cosplay)
list of tales of... characters
tales of phantasia
rondoline e. effenberg
mimi baker
tales of legendia
tales of rebirth
saleh (tales)
sol 644
dracotail arthalion
dracotail lukias
aerial battle
spread arms
backbend
sapulaisimie
tenkyuu chimata pose
umamusume: cinderella gray
ceras yanagida lilienfeld
badboon
vanilla (vanillaklein)
Options
Wiki History
Discussions
What Links Heretag group:eyes tags
[See tag groups .]
Table of Contents
Iris
Individual colors of the iris
• aqua eyes
• black eyes
• blue eyes
• brown eyes
• green eyes
• grey eyes
• orange eyes
• purple eyes
• pink eyes
• red eyes
• white eyes
• yellow eyes
Multiple colors of the iris
• heterochromia
• multicolored eyes
• gradient eyes
• two-ton

In [33]:
def clean_extracted_text(text):
    cleaned_text = ""

    for i, line in enumerate(text.split('\n')):
        if len(line) > 10 and i > 70:
            cleaned_text += line + '\n'

    cleaned_text = cleaned_text.replace('.', '')
    cleaned_text = cleaned_text.replace('~', '')
    cleaned_text = cleaned_text.replace('©', '')
    cleaned_text = cleaned_text.replace('_', '')
    cleaned_text = cleaned_text.replace(';:;', '')
    return cleaned_text



In [34]:
cleaned_text = clean_extracted_text(text)
len(cleaned_text)

3229

In [36]:


from langchain.text_splitter import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
    add_start_index=True,
)



In [41]:
texsts= text_splitter.create_documents([cleaned_text])
texsts[0].page_content

'• orange pupils\n• pink pupils\n• purple pupils\n• red pupils\n• white pupils\n• yellow pupils\nForm of the pupils\n• constricted pupils\n• dilated pupils\n• extra pupils\n• horizontal pupils\n• no pupils\n• slit pupils\n• symbol-shaped pupils\n• diamond-shaped pupils\n• flower-shaped pupils\n• heart-shaped pupils\n• star-shaped pupils\n• solid circle pupils\n• cross-shaped pupils\n• x-shaped pupils\n• snowflakes-shapedpupils\n• power symbol-shaped pupils\n• crosshair pupils\n• mismatched pupils\n• blue sclera\n• black sclera\n• blank eyes  (white sclera)\n• bloodshot eyes\n• green sclera\n• mismatched sclera\n• no sclera\n• orange sclera\n• red sclera\n• yellow sclera\nAround the eyes\n• bags under eyes\n• aegyo sal\n• bruised eye\n• flaming eyes\n• glowing eyesTag Group:Eyes Tags Wiki | Danbooru https://danboorudonmaius/wikipages/taggroup%3Aeyestags\nСтр 2 из 6 30072025, 20:08• glowing eye\nMore appearance\nAnimal or inhuman eyes\n• button eyes\n• cephalopod eyes\n• compound eyes\n•

In [42]:
documents=[]

for i in texsts:
    documents.append(i)

Changes to the new embedding models

For the new embeddings model, embedding-001, there is a new task type parameter and the optional title (only valid with task_type=RETRIEVAL_DOCUMENT).

These new parameters apply only to the newest embeddings models.The task types are:
| Task Type           | Description                                                                 |
|---------------------|-----------------------------------------------------------------------------|
| RETRIEVAL_QUERY     | Specifies the given text is a query in a search/retrieval setting.          |
| RETRIEVAL_DOCUMENT  | Specifies the given text is a document in a search/retrieval setting.       |
| SEMANTIC_SIMILARITY | Specifies the given text will be used for Semantic Textual Similarity (STS).|
| CLASSIFICATION      | Specifies that the embeddings will be used for classification.              |
| CLUSTERING          | Specifies that the embeddings will be used for clustering.                  |

In [46]:
class GeminiEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        model = 'models/embedding-001'
        # for better results, try to provide a title for each input if the corpus is covering a lot of domains
        title = "tags"

        return genai.embed_content(
            model=model,
            content=input,
            task_type="retrieval_document",
            title=title)["embedding"]

In [ ]:
import time
from tqdm import tqdm
